In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("../data/raw/carvana.csv")
df.columns = df.columns.str.strip()
df.shape

(22000, 4)

## Data Cleaning (same fixes as Assignment 2)
Before training any model, apply the same cleaning steps: strip whitespace from
`Name`, fix corrupted `Year` values, and drop exact duplicate rows.

In [2]:
df['Name'] = df['Name'].str.strip()
df['Year'] = df['Year'].astype(str).str[:4].astype(int)
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(9317, 4)

## Splitting the Data
Before training any model, split the data into a training set (the model learns
from this) and a test set (used only to evaluate the model on data it has never
seen). This is what makes the evaluation honest.

In [3]:
X = df[["Year", "Miles"]]
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 7453
Testing rows: 1864


## Model 1: Linear Regression

In [4]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Intercept (base price):", lr_model.intercept_)
print("Coefficients (Year, Miles):", lr_model.coef_)

Intercept (base price): -1594600.3425905542
Coefficients (Year, Miles): [ 8.04634286e+02 -1.15508544e-01]


## Evaluating Linear Regression

In [5]:
y_pred_lr = lr_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression MAE: ${mae_lr:,.2f}")
print(f"Linear Regression R²: {r2_lr:.4f}")

Linear Regression MAE: $4,131.52
Linear Regression R²: 0.3677


## Model 2: Random Forest

In [6]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest MAE: ${mae_rf:,.2f}")
print(f"Random Forest R²: {r2_rf:.4f}")

Random Forest MAE: $4,508.57
Random Forest R²: 0.1681


## Comparing the Two Models

In [7]:
comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE ($)": [mae_lr, mae_rf],
    "R2": [r2_lr, r2_rf],
})
comparison

,Model,MAE ($),R2
0,Linear Regression,4131.517661,0.367706
1,Random Forest,4508.572727,0.168070


## Conclusion: Which Model Predicts More Accurately, and Why?

**Linear Regression outperformed Random Forest** on this task — MAE of $4,131.52
vs $4,508.57, and R² of 0.368 vs 0.168. Linear Regression explains roughly twice as
much of the price variation as Random Forest does, and its predictions are on
average about $377 closer to the true price.

This is the opposite of what's often assumed ("more complex model = better"), and
there's a real reason for it here, not just chance:

1. **Very few features.** Only `Year` and `Miles` were used as inputs. Random
   Forest's real advantage is finding complex, non-linear interactions across many
   features — with just two fairly simple, roughly linear predictors, there isn't
   much genuine complexity for it to exploit, but there's still enough noise for
   it to latch onto and overfit.

2. **Noisy labels.** Many rows share the same `Year` and `Miles` but have
   different `Price` (different trims, different negotiated deals — something
   already visible from the cleaning work in Assignment 2). From a model's
   perspective, this looks like unpredictable noise. Random Forest's flexibility
   lets it fit that noise in training, which then hurts it on the unseen test set.
   Linear Regression's simplicity, which normally looks like a limitation,
   actually protects it here — it can only draw one straight relationship and
   can't chase noise the way a forest of deep trees can.

3. **Default tree depth.** The Random Forest was trained with default settings, so
   each tree was allowed to grow fully deep — closer to the single Decision Tree
   overfitting risk described in the lecture, before averaging (partially)
   corrects for it.

**Takeaway:** a more flexible model is not automatically a more accurate one —
it depends on whether the added flexibility matches genuine complexity in the
data. With only two fairly linear features, the simpler model won.